In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import nltk
import numpy as np
import pandas as pd
import plotly.express as px

from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec

from sklearn.metrics.pairwise import cosine_similarity

import spacy


# ============================================================
# DOWNLOAD NLTK
# ============================================================

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")
    nltk.download("punkt_tab")


# ============================================================
# LOAD SPACY
# ============================================================

# python -m spacy download pt_core_news_lg

nlp = spacy.load("pt_core_news_lg")


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def truncar_texto(texto, limite=100):

    texto = str(texto)

    if len(texto) <= limite:
        return texto

    return texto[:limite] + "..."


def vetor_medio_documento(
    tokens,
    model_w2v,
    vector_dim
):

    vetores = [

        model_w2v.wv[token]

        for token in tokens

        if token in model_w2v.wv
    ]

    if len(vetores) == 0:

        return np.zeros(vector_dim)

    return np.mean(vetores, axis=0)


def gerar_heatmap(
    documentos,
    vetores_documentos,
    indices_heatmap,
    coluna,
    titulo_modelo
):

    docs_heatmap = [
        documentos[i]
        for i in indices_heatmap
    ]

    vetores_heatmap = vetores_documentos[
        indices_heatmap
    ]

    matriz_sim = cosine_similarity(
        vetores_heatmap
    )

    labels = [
        f"{i}"
        for i in indices_heatmap
    ]

    fig = px.imshow(
        matriz_sim,

        x=labels,
        y=labels,

        color_continuous_scale="Viridis",

        text_auto=".2f",

        aspect="auto"
    )

    customdata = []

    for linha_texto in docs_heatmap:

        linha_custom = []

        for coluna_texto in docs_heatmap:

            linha_custom.append([

                truncar_texto(linha_texto, 100),

                truncar_texto(coluna_texto, 100)
            ])

        customdata.append(linha_custom)

    customdata = np.array(customdata)

    fig.update_traces(

        customdata=customdata,

        hovertemplate=
            "<b>Texto Linha:</b><br>%{customdata[0]}<br><br>" +
            "<b>Texto Coluna:</b><br>%{customdata[1]}<br><br>" +
            "<b>Similaridade:</b> %{z:.4f}<br>" +
            "<extra></extra>"
    )

    fig.update_layout(

        title=(
            f"{titulo_modelo} - "
            f"SIMILARIDADE DOCUMENTO x DOCUMENTO "
            f"({coluna})"
        ),

        height=1000,
        width=1000
    )

    fig.show()


def exibir_resultados(
    resultado,
    documentos,
    vetores_documentos,
    coluna,
    titulo_modelo,
    top_n=30
):

    # ========================================================
    # TOP MAIS SIMILARES
    # ========================================================

    top_similares = (
        resultado
        .sort_values(
            "similaridade",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    # ========================================================
    # TOP MENOS SIMILARES
    # ========================================================

    top_menos = (
        resultado
        .sort_values(
            "similaridade",
            ascending=True
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    # ========================================================
    # HEATMAP
    # ========================================================

    indices_heatmap = (

        top_similares["indice"].tolist()

        +

        top_menos["indice"].tolist()
    )

    indices_heatmap = list(
        dict.fromkeys(indices_heatmap)
    )

    gerar_heatmap(
        documentos,
        vetores_documentos,
        indices_heatmap,
        coluna,
        titulo_modelo
    )

    # ========================================================
    # TABELAS
    # ========================================================

    print("\nTOP MAIS SIMILARES")

    display(
        top_similares[
            ["indice", "similaridade", "documento"]
        ]
    )

    print("\nTOP MENOS SIMILARES")

    display(
        top_menos[
            ["indice", "similaridade", "documento"]
        ]
    )


# ============================================================
# ANÁLISE WORD2VEC
# ============================================================

def analisar_similaridade_w2v(

    df,

    colunas=["stop", "stop_lemma", "stop_stem"],

    top_n=30,

    vector_dim=100,

    window_size=5,

    min_word_count=1,

    sg=0,

    random_state=42
):

    print("\n" + "=" * 100)
    print("ANÁLISE COM WORD2VEC")
    print("=" * 100)

    for coluna in colunas:

        print("\n" + "=" * 90)
        print(f"COLUNA: {coluna}")
        print("=" * 90)

        # ====================================================
        # PREPARAÇÃO
        # ====================================================

        dados = (
            df[coluna]
            .fillna("")
            .astype(str)
            .reset_index(drop=True)
        )

        dados = dados[
            dados.str.strip() != ""
        ].reset_index(drop=True)

        if len(dados) < 2:

            print(
                f"Coluna {coluna} possui poucos dados."
            )

            continue

        documentos = dados.tolist()

        query = documentos[0]

        print("\nQUERY:")
        print(query)

        # ====================================================
        # TOKENIZAÇÃO
        # ====================================================

        documentos_tokenizados = [

            word_tokenize(doc.lower())

            for doc in documentos
        ]

        # ====================================================
        # TREINAMENTO WORD2VEC
        # ====================================================

        model_w2v = Word2Vec(

            sentences=documentos_tokenizados,

            vector_size=vector_dim,

            window=window_size,

            min_count=min_word_count,

            workers=4,

            sg=sg,

            seed=random_state
        )

        print(
            f"\nVocabulário: "
            f"{len(model_w2v.wv.index_to_key)} palavras"
        )

        # ====================================================
        # VETORES DOCUMENTOS
        # ====================================================

        vetores_documentos = np.array([

            vetor_medio_documento(
                tokens,
                model_w2v,
                vector_dim
            )

            for tokens in documentos_tokenizados
        ])

        # ====================================================
        # QUERY
        # ====================================================

        query_tokens = word_tokenize(
            query.lower()
        )

        vetor_query = vetor_medio_documento(
            query_tokens,
            model_w2v,
            vector_dim
        )

        # ====================================================
        # SIMILARIDADE
        # ====================================================

        similaridades = cosine_similarity(

            vetor_query.reshape(1, -1),

            vetores_documentos

        )[0]

        resultado = pd.DataFrame({

            "indice": range(len(documentos)),

            "documento": documentos,

            "similaridade": similaridades
        })

        # remove a própria query
        resultado = resultado[
            resultado["indice"] != 0
        ]

        # ====================================================
        # RESULTADOS
        # ====================================================

        exibir_resultados(
            resultado,
            documentos,
            vetores_documentos,
            coluna,
            titulo_modelo="WORD2VEC",
            top_n=top_n
        )


# ============================================================
# ANÁLISE SPACY
# ============================================================

def analisar_similaridade_spacy(

    df,

    colunas=["stop", "stop_lemma", "stop_stem"],

    top_n=30
):

    print("\n" + "=" * 100)
    print("ANÁLISE COM SPACY")
    print("=" * 100)

    for coluna in colunas:

        print("\n" + "=" * 90)
        print(f"COLUNA: {coluna}")
        print("=" * 90)

        # ====================================================
        # PREPARAÇÃO
        # ====================================================

        dados = (
            df[coluna]
            .fillna("")
            .astype(str)
            .reset_index(drop=True)
        )

        dados = dados[
            dados.str.strip() != ""
        ].reset_index(drop=True)

        if len(dados) < 2:

            print(
                f"Coluna {coluna} possui poucos dados."
            )

            continue

        documentos = dados.tolist()

        query = documentos[0]

        print("\nQUERY:")
        print(query)

        # ====================================================
        # VETORES SPACY
        # ====================================================

        vetores_documentos = np.array([

            nlp(doc.lower()).vector

            for doc in documentos
        ])

        vetor_query = nlp(
            query.lower()
        ).vector

        print(
            f"\nDimensão vetores spaCy: "
            f"{len(vetor_query)}"
        )

        # ====================================================
        # SIMILARIDADE
        # ====================================================

        similaridades = cosine_similarity(

            vetor_query.reshape(1, -1),

            vetores_documentos

        )[0]

        resultado = pd.DataFrame({

            "indice": range(len(documentos)),

            "documento": documentos,

            "similaridade": similaridades
        })

        # remove a própria query
        resultado = resultado[
            resultado["indice"] != 0
        ]

        # ====================================================
        # RESULTADOS
        # ====================================================

        exibir_resultados(
            resultado,
            documentos,
            vetores_documentos,
            coluna,
            titulo_modelo="SPACY",
            top_n=top_n
        )


# ============================================================
# EXECUÇÃO
# ============================================================

# Exemplo:
#
# analisar_similaridade_w2v(
#     df,
#     colunas=["stop"],
#     top_n=20
# )
#
# analisar_similaridade_spacy(
#     df,
#     colunas=["stop"],
#     top_n=20
# )


# ============================================================
# EXEMPLO COMPLETO
# ============================================================

# WORD2VEC
analisar_similaridade_w2v(
    df,
    colunas=["stop"],
    top_n=20
)

# SPACY
analisar_similaridade_spacy(
    df,
    colunas=["stop"],
    top_n=20
)